# polygon → MOC → shard → 3-D → numpy

The whole zagg read stack in five functions, two `pip install`s, and zero
credentials. A geojson polygon becomes a morton MOC; the MOC checks itself
against the store's own coverage; the covered shards open with timings; one
shard renders in 3-D (ATL03 + GEDI together); the current view exports to a
numpy tensor and saves to disk. Everything below is reader-side — `mortie`
for the geometry, `moczarr` for the store — running anonymously against
public S3, binder-ready.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipympl ipywidgets
%matplotlib widget

import resource
import time

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Checkbox, Dropdown, HBox, VBox, interactive_output
from matplotlib.colors import LogNorm

import moczarr as mz
from moczarr.hhdc import block_rank, rank_to_rowcol  # the reader's layout kernel
from mortie import moc, toc2time

# One store per product, each appendable. Coverage answers which ground the
# store holds; the store name never does.
STORES = {
    "atl03": ("s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr", "19/h_tdigest_signal"),
    "gedi": ("s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr", "18/rx_flux"),
}
S3 = {"region": "us-west-2", "anonymous": True}

BLOCK_ORDER = 12
# o12 block side (equal-area, square-equivalent) -- the frame the 3-D view plots
# in, and the same square the exported tensors tile.
_SIDE12 = float(np.sqrt(4 * np.pi / (12 * 4**BLOCK_ORDER)) * 6_371_000)
_CAP = 20_000  # points drawn per sensor per block; a leaf holds millions

rss = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20  # noqa: E731

## One polygon in, covered shards out

In [ ]:
aoi = {"features": [{"geometry": {"coordinates": [[  # a ~4 km box on the SERC tract
    [-76.56, 38.87], [-76.50, 38.87], [-76.50, 38.91], [-76.56, 38.91], [-76.56, 38.87]
]]}}]}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not cover the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)
shards

## Open one shard — every dataset, timed

In [ ]:
def open_shard(shard):
    """Open the shard's leaf in every store; print open+sweep timing and RSS."""
    handles = {}
    for name, (root, field) in STORES.items():
        t0, r0 = time.perf_counter(), rss()
        store = mz.open_leaf(root, shard, **S3)
        _, element = mz.open_ragged(store, field)
        n = sum(len(v) for _, v in mz.read_ragged(store, field))
        print(f"{name:6s} open+sweep {time.perf_counter() - t0:5.1f}s "
              f"(+{rss() - r0:4.0f} MB) — {n:,} centroids, element {element}")
        handles[name] = (store, field)
    return handles


handles = open_shard(shards[0])

## The 3-D view — both sensors, exact centroids, time-aware

In [ ]:
def _grid_xy(words, block_order=BLOCK_ORDER):
    """Word -> (x, y) meters inside its o12 block. Three library calls, no bits.

    `block_rank` recovers each word's block-local nested rank and its own
    order -- normalizing POINT words to their order-29 area twins on the way,
    which is the step whose absence used to make the level-28/29 digits decode
    out of range. `rank_to_rowcol` is the bit deinterleave (mortie spec
    section 8): it returns `(row, col) = (y, x)` with `[0, 0]` at the
    subtree's south corner, so x reads the row and y the col.

    A leaf mixes orders -- o29 located words beside coarser merged-centroid
    words, and GEDI's cell words are o18 -- so the ranks are grouped by depth
    and handed over one vectorized call per depth.
    """
    rank, order = block_rank(words, block_order)
    depth = order - block_order
    row = np.zeros(len(rank)); col = np.zeros(len(rank))
    for d in np.unique(depth):
        m = np.flatnonzero(depth == d)
        r, c = rank_to_rowcol(rank[m], int(d))
        row[m] = np.asarray(r, dtype=float); col[m] = np.asarray(c, dtype=float)
    side = 2.0 ** depth.astype(float)  # cells along a block edge
    return (row + 0.5) / side * _SIDE12, (col + 0.5) / side * _SIDE12

def _load(store, field):
    """One sensor's centroids: z, weight, xy, o12 block, acquisition days.

    xy comes from the located sibling's word where the field declares one
    (ATL03 -- point-exact) and from the cell word where it does not (GEDI
    flux, whose shots are unlocated by design, so a centroid renders at its
    cell's center). Both land in the block's own lattice frame.
    """
    arr, _ = mz.open_ragged(store, field)
    attrs = dict(arr.attrs)
    locname = (attrs.get("ragged") or {}).get("locations")
    tname = attrs.get("times")
    zs, wts, cells, locw, seq = [], [], [], [], []
    for row in mz.read_ragged(store, field, locations=bool(locname)):
        seq.append(row[0])
        v = np.asarray(row[1])
        zs.append(v[:, 0]); wts.append(v[:, 1])
        cells.append(np.full(len(v), row[0], dtype=np.uint64))
        if locname:
            locw.append(np.asarray(row[2], dtype=np.uint64))
    z, wt = np.concatenate(zs), np.concatenate(wts)
    cells = np.concatenate(cells)
    sh = np.uint64(6 + 2 * (27 - BLOCK_ORDER))
    blocks = ((cells >> sh) << sh) | np.uint64(BLOCK_ORDER)  # o12 ancestor by truncation
    x, y = _grid_xy(np.concatenate(locw) if locname else cells)
    t = None
    if tname:
        tmap = {int(r[0]): np.asarray(r[1], dtype=np.uint64).ravel()
                for r in mz.read_ragged(store, field.rsplit("/", 1)[0] + "/" + tname)}
        tw = np.concatenate([tmap[int(c)] for c in seq])
        ns2018 = float((np.datetime64("2018-01-01") - np.datetime64("1850-01-01"))
                       // np.timedelta64(1, "ns"))
        t = (np.asarray(toc2time(tw)[0], dtype="float64") - ns2018) / 86.4e12
    return {"z": z, "wt": wt, "x": x, "y": y, "blocks": blocks, "t": t,
            "xy_note": "exact xy" if locname else "cell-center xy"}


class View:
    """Holds what is on screen so `export` knows which block you mean."""
    shard = None
    block = None


def view3d(handles, shard):
    """Interactive paired 3-D view. Drag to rotate -- the two panes stay linked."""
    view = View()
    view.shard = shard
    data = {name: _load(store, field) for name, (store, field) in handles.items()}
    names = list(data)

    # Blocks both sensors populate, SORTED, each labelled with how much it holds.
    joint = sorted(set.intersection(*(set(np.unique(d["blocks"]).tolist()) for d in data.values())))
    counts = {w: {n: int((data[n]["blocks"] == np.uint64(w)).sum()) for n in names} for w in joint}
    options = [(f"{mz.morton_decimal(w)}  ("
                + ", ".join(f"{counts[w][n]:,} {n}" for n in names) + ")", w) for w in joint]

    # Subsample per block once, deterministically, so rotation stays smooth.
    rng = np.random.default_rng(0)
    picks = {w: {n: (lambda m: m if len(m) <= _CAP else m[rng.choice(len(m), _CAP, replace=False)])(
        np.flatnonzero(data[n]["blocks"] == np.uint64(w))) for n in names} for w in joint}

    dd = Dropdown(options=options, value=joint[0], description="block")
    zmode = Dropdown(options=[("independent z", "auto"), *[(f"pin z to {n}", n) for n in names]],
                     value="auto", description="z extent")
    elev_cb = Checkbox(value=False, description="color by elevation (shared)")
    time_cb = Checkbox(value=False, description="color by time (shared)")

    def draw(block, by_elev, by_time, zmode):
        view.block = block
        panes = [{**{k: data[n][k][picks[block][n]] for k in ("z", "wt", "x", "y")},
                  "t": None if data[n]["t"] is None else data[n]["t"][picks[block][n]],
                  "label": n, "xy_note": data[n]["xy_note"]} for n in names]
        zlim = None if zmode == "auto" else (
            lambda p: (p["z"].min(), p["z"].max()))(panes[names.index(zmode)])
        shared_elev = (plt.Normalize(min(p["z"].min() for p in panes),
                                     max(p["z"].max() for p in panes))
                       if by_elev and not by_time else None)
        tv = [p["t"] for p in panes if p["t"] is not None]
        shared_time = plt.Normalize(min(t.min() for t in tv), max(t.max() for t in tv)) if (by_time and tv) else None
        ticks = [0.0, _SIDE12 / 2, _SIDE12]
        labels = [f"{v:.0f} m" for v in ticks]

        fig = plt.figure(figsize=(11, 5.2))
        axes = []
        for k, (p, cmap) in enumerate(zip(panes, ("viridis", "plasma"))):
            ax = fig.add_subplot(1, 2, k + 1, projection="3d")
            axes.append(ax)
            note = f" — {p['xy_note']}"
            alpha = np.clip(p["wt"] / max(np.percentile(p["wt"], 98), 1e-9), 0.08, 1.0)
            if by_time and p["t"] is not None:
                pts = ax.scatter(p["x"], p["y"], p["z"], c=p["t"], s=1.5, cmap="turbo",
                                 norm=shared_time, alpha=alpha)
                fig.colorbar(pts, shrink=0.55, pad=0.10, label="days since 2018-01-01")
            elif by_time:
                ax.scatter(p["x"], p["y"], p["z"], color="#9498a0", s=1.5, alpha=0.15)
                note += " — no temporal channel"
            elif by_elev:
                pts = ax.scatter(p["x"], p["y"], p["z"], c=p["z"], s=1.5, cmap="viridis",
                                 norm=shared_elev, alpha=alpha)
                fig.colorbar(pts, shrink=0.55, pad=0.10, label="elevation (m)")
            else:
                pts = ax.scatter(p["x"], p["y"], p["z"], c=p["wt"], s=1.5, cmap=cmap,
                                 norm=LogNorm(), alpha=alpha)
                fig.colorbar(pts, shrink=0.55, pad=0.10, label="weight")
            if zlim is not None:
                ax.set_zlim(*zlim)
            ax.set_title(p["label"] + note, fontsize=9)
            ax.set_zlabel("elevation (m)")
            ax.set_xticks(ticks, labels, fontsize=7)
            ax.set_yticks(ticks, labels, fontsize=7)
            ax.set_xlabel("east", fontsize=8, labelpad=-2)
            ax.set_ylabel("north", fontsize=8, labelpad=-2)

        # Linked rotation: only while DRAGGING, and only when the angles moved --
        # a bare hover must not trigger a redraw.
        def _sync(event):
            if event.button is None or event.inaxes not in axes:
                return
            src = event.inaxes
            for other in axes:
                if other is not src and (other.elev != src.elev or other.azim != src.azim):
                    other.view_init(elev=src.elev, azim=src.azim)
                    fig.canvas.draw_idle()

        fig.canvas.mpl_connect("motion_notify_event", _sync)
        fig.suptitle(f"shard {shard} — block {mz.morton_decimal(block)} — exact centroids",
                     fontsize=11)
        plt.show()

    out = interactive_output(draw, {"block": dd, "by_elev": elev_cb, "by_time": time_cb,
                                    "zmode": zmode})
    display(VBox([HBox([dd, zmode]), HBox([elev_cb, time_cb]), out]))
    return view


view = view3d(handles, shards[0])

## Export what you see — a numpy tensor, shaped your way

In [ ]:
def export(view, sensor, n_bins=64, resolution=1.0, fit="degrade_resolution"):
    """The block on screen -> (rows, cols, n_bins) numpy tensor + z metadata.

    `n_bins`/`resolution` set the vertical shape; the xy shape follows the
    sensor's cell order within the o12 block -- 2**(cell_order - 12) a side,
    so 128 for ATL03's o19 cells and 64 for GEDI's o18.
    """
    store, field = handles[sensor]
    t, mask, (z0, dz), w = next(
        b for b in mz.read_tensors(store, field, n_bins=n_bins, resolution=resolution,
                                   block_order=BLOCK_ORDER, fit=fit)
        if int(b[3]) == int(view.block)
    )
    print(f"{sensor}: {t.shape} tensor, z = {z0:.1f} m + bin * {dz:g} m")
    return t, {"z0": z0, "dz": dz, "block": mz.morton_decimal(w)}


gedi, meta = export(view, "gedi", n_bins=128, resolution=0.5)
np.save(f"gedi_{meta['block']}.npy", gedi)

atl03, meta = export(view, "atl03")           # default 64 x 1 m bins
np.save(f"atl03_{meta['block']}.npy", atl03)

In [ ]:
# ---- second export: every block of the shard as a 128 x 128 x 128 cube ----
# ATL03's cells are o19, so an o12 block is 2**(19-12) = 128 cells a side;
# ask for 128 z-bins at 1 m and the tensor is a cube. One `read_tensors`
# sweep yields one cube per populated block, so a shard exports a SET.
#
# `fit="degrade_resolution"` keeps a block whose trimmed relief exceeds 128 m
# rather than refusing it -- the z gain comes back per block, so the ones that
# did not fit at 1 m say so instead of silently pretending.
cubes = {}
for tensor, mask, (z0, dz), w in mz.read_tensors(
    *handles["atl03"], n_bins=128, resolution=1.0, block_order=BLOCK_ORDER,
    fit="degrade_resolution",
):
    cubes[mz.morton_decimal(w)] = {"tensor": tensor, "z0": z0, "dz": dz,
                                   "cells": int(mask.astype(bool).sum())}

exact = [b for b, c in cubes.items() if c["dz"] == 1.0]
print(f"{len(cubes)} blocks -> {len(cubes)} cubes of {next(iter(cubes.values()))['tensor'].shape}")
print(f"  {len(exact)} at the requested 1 m bins; "
      f"{len(cubes) - len(exact)} degraded to fit their relief in 128 bins")
for b, c in sorted(cubes.items())[:5]:
    print(f"  {b}  z = {c['z0']:8.1f} m + bin * {c['dz']:.2f} m   "
          f"{c['cells']:5,} populated cells   {c['tensor'].nbytes / 2**20:.1f} MiB")

np.savez_compressed(
    f"atl03_cubes_{shards[0]}.npz",
    **{b: c["tensor"] for b, c in cubes.items()},
)
print(f"saved atl03_cubes_{shards[0]}.npz")

Five functions, two libraries, one polygon — coverage, shards, timings,
the paired 3-D view, and tensors on disk.